# Benchmark continuite d'identite (tracking)

Compare **ByteTrack** vs **BoT-SORT + ReID** sur des clips handball, avec le
modele fine-tune. Produit : videos annotees (IDs colores + coupures de plan),
`comparatif.csv/json` et `RAPPORT.md`.

**Prerequis** : runtime **T4 GPU**, et les 3 clips deposes sur Drive dans
`/MyDrive/PIVOT_AI/benchmark/`.

## 1. Clone (branche benchmark) + install

In [ ]:
import os, subprocess, sys

REPO_OWNER = "tristan-paloc"
REPO_NAME  = "pivot-ai-handball"
BRANCHE    = "feat/benchmark-tracking"
REPO_DIR   = f"/content/{REPO_NAME}"
REPO_URL   = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

if not os.path.exists(REPO_DIR):
    res = subprocess.run(["git", "clone", "--branch", BRANCHE, "--depth", "1",
                          REPO_URL, REPO_DIR], capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(res.stderr)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCHE],
                   capture_output=True, text=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCHE],
                   capture_output=True, text=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True)

%cd {REPO_DIR}
!pip install -q -e .
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
import pivot_ai; print(f"pivot_ai version : {pivot_ai.__version__}")

## 2. Verifier le GPU

In [ ]:
import torch
print(f"CUDA dispo : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("Active le GPU : Execution > Modifier le type d'execution > T4 GPU")

## 3. Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DOSSIER_CLIPS  = "/content/drive/MyDrive/PIVOT_AI/benchmark"
DOSSIER_SORTIE = "/content/drive/MyDrive/PIVOT_AI/benchmark_out"
MODELE         = "/content/drive/MyDrive/PIVOT_AI/models/handball_yolov8m.pt"

assert os.path.isdir(DOSSIER_CLIPS), f"Depose les clips dans {DOSSIER_CLIPS}"
assert os.path.exists(MODELE), f"Modele introuvable : {MODELE}"
!ls -la {DOSSIER_CLIPS}

## 4. Lancer le benchmark

ByteTrack vs BoT-SORT sur les 3 clips. Compter quelques minutes sur T4
(BoT-SORT/ReID est le plus lent).

In [ ]:
!python -m pivot_ai.cli benchmark --clips {DOSSIER_CLIPS} --sortie {DOSSIER_SORTIE} --modele {MODELE} --trackers bytetrack,botsort --subsample 2

## 5. Comparatif chiffre

In [ ]:
import polars as pl
df = pl.read_csv(f"{DOSSIER_SORTIE}/comparatif.csv")
print(df)
print("\n=== RAPPORT ===\n")
print(open(f"{DOSSIER_SORTIE}/RAPPORT.md", encoding="utf-8").read())

## 6. Verification visuelle

Quelques frames d'une video annotee (couleur = ID ; si la couleur d'un joueur
change sans coupure de plan, c'est une casse d'ID).

In [ ]:
import glob
import cv2
import matplotlib.pyplot as plt

videos = sorted(glob.glob(f"{DOSSIER_SORTIE}/*.mp4"))
print("Videos annotees :")
for v in videos:
    print("  ", v)

if videos:
    cap = cv2.VideoCapture(videos[0])
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    for ax, frac in zip(axes.flat, [0.2, 0.45, 0.7, 0.9]):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(n * frac))
        ok, frame = cap.read()
        if ok:
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax.set_title(os.path.basename(videos[0]))
            ax.axis('off')
    cap.release()
    plt.tight_layout(); plt.show()